# Comparación de Estrategias Federadas vs Proactive Forest Base (Nayma)

## Objetivo
Comparar las métricas (Accuracy y F1) de cada estrategia federada (S1-S7 + PW) con los resultados base de Proactive Forest reportados por Nayma.

## Configuración
- **n_clients = 3** (se promedian las métricas de los 3 clientes)
- **seed = 42**
- **n_estimators = 100**
- **distribution = iid**
- **alpha_pf = 0.45**

## Datasets
Car, Iris, Letter, Nursery, Optdigits, Sonar, Spambase, Vowel

## Estructura
- Cada estrategia (S1-S8 y PW) tiene su propia celda independiente
- Se calculan métricas por cliente (Accuracy y F1)
- Se promedian las métricas de los 3 clientes
- Tabla final comparativa con los resultados de Nayma

---
## 0. Imports y configuración común

In [ ]:
import sys
from pathlib import Path

# Resolve project root: go up from notebooks/ directory
NOTEBOOK_DIR = Path(__file__).resolve().parent if '__file__' in dir() else Path.cwd()
ROOT = NOTEBOOK_DIR.parent.parent.parent  # Adjust to reach project root

# Fallback: try common paths
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]:
    if (candidate / 'src').exists() and (candidate / 'data').exists():
        ROOT = candidate
        break

sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT}")

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from src.application.orchestrators.fl_orchestrator import FLEXOrchestrator
from src.domain.dataset.base_adapter import DatasetSplit
from src.infrastructure.dataset.csv_adapter import GenericCsvAdapter

# ── Fixed parameters ─────────────────────────────────────────────────────
SEED = 42
N_CLIENTS = 3
N_ESTIMATORS = 100
ALPHA_PF = 0.45
T_MAX = 100
LOCAL_WEIGHT = 0.5
F1_WEIGHT = 0.6

np.random.seed(SEED)

---
## 1. Dataset Loaders

In [ ]:
def load_dataset_from_adapter(name, file_path, target_column, categorical_features=None):
    """Generic loader with stratified split to ensure all classes in test set."""
    df = pd.read_csv(ROOT / file_path)
    fcols = [c for c in df.columns if c != target_column]
    
    # Encode categorical features
    if categorical_features:
        enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        df[categorical_features] = enc.fit_transform(df[categorical_features])
    
    X = df[fcols].values.astype(float)
    
    # Encode target
    le = LabelEncoder()
    y = le.fit_transform(df[target_column])
    
    # Stratified split to ensure all classes in both sets
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
    
    # Scale
    sc = StandardScaler()
    X_tr = sc.fit_transform(X_tr)
    X_te = sc.transform(X_te)
    
    # Convert to strings for consistency with class_names
    y_tr_str = np.array([le.classes_[i] for i in y_tr])
    y_te_str = np.array([le.classes_[i] for i in y_te])
    
    return DatasetSplit(
        X_train=X_tr, y_train=y_tr_str, X_test=X_te, y_test=y_te_str,
        feature_names=fcols, class_names=list(le.classes_), dataset_name=name
    )

def load_car():
    ds = load_dataset_from_adapter('car', 'data/car.csv', 'class', 
                                   categorical_features=['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety'])
    # Ensure y_test contains all classes
    _verify_classes(ds, 'car')
    return ds

def load_iris():
    from sklearn.datasets import load_iris as sk_load_iris
    data = sk_load_iris()
    X_tr, X_te, y_tr, y_te = train_test_split(data.data, data.target, test_size=.2, random_state=SEED, stratify=data.target)
    sc = StandardScaler()
    X_tr = sc.fit_transform(X_tr)
    X_te = sc.transform(X_te)
    # Convert numeric labels to actual class names
    y_tr_str = np.array([data.target_names[i] for i in y_tr])
    y_te_str = np.array([data.target_names[i] for i in y_te])
    ds = DatasetSplit(
        X_train=X_tr, y_train=y_tr_str, X_test=X_te, y_test=y_te_str,
        feature_names=list(data.feature_names), class_names=list(data.target_names), dataset_name='iris'
    )
    _verify_classes(ds, 'iris')
    return ds

def load_letter():
    ds = load_dataset_from_adapter('letter', 'data/letter.csv', 'class')
    _verify_classes(ds, 'letter')
    return ds

def load_nursery():
    ds = load_dataset_from_adapter('nursery', 'data/nursery.csv', 'class',
                                   categorical_features=['parents', 'has_nurs', 'form', 'children', 'housing', 'finance', 'social', 'health'])
    _verify_classes(ds, 'nursery')
    return ds

def load_optdigits():
    ds = load_dataset_from_adapter('optdigits', 'data/optdigits.csv', 'class')
    _verify_classes(ds, 'optdigits')
    return ds

def load_sonar():
    ds = load_dataset_from_adapter('sonar', 'data/sonar.csv', 'Class')
    _verify_classes(ds, 'sonar')
    return ds

def load_spambase():
    ds = load_dataset_from_adapter('spambase', 'data/spambase.csv', 'class')
    _verify_classes(ds, 'spambase')
    return ds

def load_vowel():
    df = pd.read_csv(ROOT / 'data' / 'vowel.csv')
    drop_cols = ['Train or Test', 'Speaker Number', 'Sex']
    use_cols = [c for c in df.columns if c not in drop_cols and c != 'Class']
    X = df[use_cols].values.astype(float)
    le = LabelEncoder()
    y = le.fit_transform(df['Class'])
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.2, random_state=SEED, stratify=y)
    sc = StandardScaler()
    X_tr = sc.fit_transform(X_tr)
    X_te = sc.transform(X_te)
    # Convert numeric labels back to actual class names
    y_tr_str = np.array([le.classes_[i] for i in y_tr])
    y_te_str = np.array([le.classes_[i] for i in y_te])
    ds = DatasetSplit(
        X_train=X_tr, y_train=y_tr_str, X_test=X_te, y_test=y_te_str,
        feature_names=use_cols, class_names=list(le.classes_), dataset_name='vowel'
    )
    _verify_classes(ds, 'vowel')
    return ds

def _verify_classes(ds, name):
    """Verify that y_test contains all classes. If not, raise an error with details."""
    unique_test = set(str(y) for y in ds.y_test)
    unique_all = set(str(c) for c in ds.class_names)
    missing = unique_all - unique_test
    if missing:
        raise ValueError(f"Dataset '{name}': y_test is missing classes {missing}. "
                        f"This will cause confusion_matrix to fail. "
                        f"Consider using a different seed or test_size.")
    print(f"  ✓ {name}: {len(ds.class_names)} classes, all present in test set")

DATASETS = {
    'car': load_car,
    'iris': load_iris,
    'letter': load_letter,
    'nursery': load_nursery,
    'optdigits': load_optdigits,
    'sonar': load_sonar,
    'spambase': load_spambase,
    'vowel': load_vowel,
}

---
## 2. Funciones auxiliares

In [ ]:
def build_config(strategy, n_clients=N_CLIENTS, n_estimators=N_ESTIMATORS, seed=SEED):
    """Build configuration dict for any strategy."""
    cfg = {
        'federation': {'n_clients': n_clients, 'distribution': 'iid', 'seed': seed},
        'model': {
            'n_estimators': n_estimators, 'alpha': ALPHA_PF,
            'split_criterion': 'entropy', 'use_progressive_stopping': True,
            'convergence': 0.002, 'episode_size': 5, 'verbose': False,
        },
        'aggregation': {'strategy': strategy, 't_max': T_MAX},
        'prediction': {'local_weight': LOCAL_WEIGHT, 'global_weight': 1.0 - LOCAL_WEIGHT},
        'verbose': False, 'seed': seed,
    }

    # S4, S7, PW → f1_weight / pcd_weight
    if strategy in ('S4', 'S7', 'PW'):
        cfg['aggregation']['f1_weight'] = F1_WEIGHT
        cfg['aggregation']['pcd_weight'] = 1.0 - F1_WEIGHT

    # PW → exclusive parameters
    if strategy == 'PW':
        cfg['aggregation']['window_size'] = 7
        cfg['aggregation']['max_rounds'] = 15
        cfg['aggregation']['convergence_threshold'] = 0.002

    return cfg


def calculate_client_metrics(predictions, y_true):
    """Calculate Accuracy and F1 for a client's predictions."""
    acc = accuracy_score(y_true, predictions)
    f1 = f1_score(y_true, predictions, average='macro', zero_division=0)
    return {'accuracy': acc, 'f1': f1}


def average_client_metrics(client_metrics_dict):
    """Average metrics across all clients."""
    accs = [m['accuracy'] for m in client_metrics_dict.values()]
    f1s = [m['f1'] for m in client_metrics_dict.values()]
    return {'accuracy': np.mean(accs), 'f1': np.mean(f1s)}


def run_experiment(dataset_name, strategy):
    """Run a dataset × strategy combination and return metrics per client."""
    ds = DATASETS[dataset_name]()
    cfg = build_config(strategy)
    np.random.seed(SEED)

    orch = FLEXOrchestrator.from_config(cfg)
    orch.setup_federation(ds, seed=SEED)
    results = orch.run_federated_round()

    # Per-client hybrid metrics
    client_metrics = {}
    for cid, preds in results.client_hybrid_predictions.items():
        # Ensure both y_test and predictions are the same type (strings)
        y_test_str = np.array([str(y) for y in results.y_test])
        preds_str = np.array([str(p) for p in preds])
        client_metrics[cid] = calculate_client_metrics(preds_str, y_test_str)

    return {
        'dataset': dataset_name,
        'strategy': strategy,
        'global_accuracy': results.global_accuracy,
        'global_macro_f1': results.global_macro_f1,
        'client_metrics': client_metrics,
        'avg_client_metrics': average_client_metrics(client_metrics),
        'n_trees_global': results.n_trees_global,
    }

---
## 3. Diccionario global para almacenar resultados

In [ ]:
# Global results dictionary
all_results = {}  # {(dataset, strategy): result_dict}

---
## 4. Ejecutar cada estrategia (celda independiente)

In [ ]:
# ── S1: Simple Pool ──────────────────────────────────────────────────────
STRATEGY = 'S1'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    avg = res['avg_client_metrics']
    print(f"Acc={avg['accuracy']:.4f}, F1={avg['f1']:.4f}")
print(f"✅ {STRATEGY} completada.")

In [ ]:
# ── S2: Global Accuracy ──────────────────────────────────────────────────
STRATEGY = 'S2'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    avg = res['avg_client_metrics']
    print(f"Acc={avg['accuracy']:.4f}, F1={avg['f1']:.4f}")
print(f"✅ {STRATEGY} completada.")

In [ ]:
# ── S3: Global Macro-F1 ──────────────────────────────────────────────────
STRATEGY = 'S3'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    avg = res['avg_client_metrics']
    print(f"Acc={avg['accuracy']:.4f}, F1={avg['f1']:.4f}")
print(f"✅ {STRATEGY} completada.")

In [ ]:
# ── S4: Global F1 + PCD ──────────────────────────────────────────────────
STRATEGY = 'S4'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    avg = res['avg_client_metrics']
    print(f"Acc={avg['accuracy']:.4f}, F1={avg['f1']:.4f}")
print(f"✅ {STRATEGY} completada.")

In [ ]:
# ── S5: Per-Client Accuracy ──────────────────────────────────────────────
STRATEGY = 'S5'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    avg = res['avg_client_metrics']
    print(f"Acc={avg['accuracy']:.4f}, F1={avg['f1']:.4f}")
print(f"✅ {STRATEGY} completada.")

In [ ]:
# ── S6: Per-Client Macro-F1 ──────────────────────────────────────────────
STRATEGY = 'S6'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    avg = res['avg_client_metrics']
    print(f"Acc={avg['accuracy']:.4f}, F1={avg['f1']:.4f}")
print(f"✅ {STRATEGY} completada.")

In [ ]:
# ── S7: Per-Client F1 + PCD ──────────────────────────────────────────────
STRATEGY = 'S7'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    avg = res['avg_client_metrics']
    print(f"Acc={avg['accuracy']:.4f}, F1={avg['f1']:.4f}")
print(f"✅ {STRATEGY} completada.")

In [ ]:
# ── PW: Progressive Windows ──────────────────────────────────────────────
STRATEGY = 'PW'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    avg = res['avg_client_metrics']
    print(f"Acc={avg['accuracy']:.4f}, F1={avg['f1']:.4f}")
print(f"✅ {STRATEGY} completada.")

---
## 5. Guardar resultados intermedios en CSV

In [ ]:
# Save intermediate results
out_dir = ROOT / 'results' / 'results_for_meet'
out_dir.mkdir(parents=True, exist_ok=True)

intermediate_rows = []
for (ds, strat), res in all_results.items():
    avg = res['avg_client_metrics']
    for cid, metrics in res['client_metrics'].items():
        intermediate_rows.append({
            'dataset': ds,
            'strategy': strat,
            'client_id': cid,
            'accuracy': metrics['accuracy'],
            'f1': metrics['f1'],
        })

df_intermediate = pd.DataFrame(intermediate_rows)
df_intermediate.to_csv(out_dir / 'intermediate_results.csv', index=False)
print(f"✅ Intermediate results saved to: {out_dir / 'intermediate_results.csv'}")

---
## 6. Tabla comparativa final vs Nayma

In [ ]:
# ── Nayma's baseline results (Proactive Forest) ──────────────────────────
NAYMA_RESULTS = {
    'car':       {'accuracy': 0.976625780, 'f1': 0.945782},
    'iris':      {'accuracy': 0.956000000, 'f1': 0.954981},
    'letter':    {'accuracy': 0.965045493, 'f1': 0.965253},
    'nursery':   {'accuracy': 0.995910870, 'f1': 0.954848},
    'optdigits': {'accuracy': 0.983235639, 'f1': 0.982218},
    'sonar':     {'accuracy': 0.848298701, 'f1': 0.823483},
    'spambase':  {'accuracy': 0.953879759, 'f1': 0.952755},
    'vowel':     {'accuracy': 0.971919192, 'f1': 0.968468},
}

STRATEGIES = ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'PW']
DATASET_ORDER = ['car', 'iris', 'letter', 'nursery', 'optdigits', 'sonar', 'spambase', 'vowel']

In [ ]:
# ── Build comparison table ───────────────────────────────────────────────
rows = []

for ds in DATASET_ORDER:
    row = {'BD': ds.capitalize()}

    # PF column (Nayma)
    row['Accuracy_PF'] = round(NAYMA_RESULTS[ds]['accuracy'], 6)
    row['F1_PF'] = round(NAYMA_RESULTS[ds]['f1'], 6)

    # Strategy columns
    for strat in STRATEGIES:
        key = (ds, strat)
        if key in all_results:
            res = all_results[key]
            avg = res['avg_client_metrics']
            row[f'Accuracy_{strat}'] = round(avg['accuracy'], 6)
            row[f'F1_{strat}'] = round(avg['f1'], 6)
        else:
            row[f'Accuracy_{strat}'] = None
            row[f'F1_{strat}'] = None

    rows.append(row)

df_comparison = pd.DataFrame(rows)

# Reorder columns: BD, PF, then each strategy
col_order = ['BD', 'Accuracy_PF', 'F1_PF']
for strat in STRATEGIES:
    col_order += [f'Accuracy_{strat}', f'F1_{strat}']
df_comparison = df_comparison[col_order]

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', lambda x: f'{x:.6f}' if pd.notna(x) else '')

print("\n" + "="*80)
print("📊 COMPARISON TABLE: Federated Strategies vs Nayma's Proactive Forest")
print("="*80)
print(df_comparison.to_string(index=False))

In [ ]:
# ── Save comparison table ────────────────────────────────────────────────
df_comparison.to_csv(out_dir / 'comparison_table.csv', index=False)
try:
    df_comparison.to_excel(out_dir / 'comparison_table.xlsx', index=False)
    print(f"\n✅ Tables saved in: {out_dir}")
except Exception as e:
    print(f"\n✅ CSV saved. Excel skipped: {e}")

---
## 7. Tabla de diferencias vs Nayma (estrategia - PF)

In [ ]:
# ── Differences table (strategy - PF) ────────────────────────────────────
diff_rows = []

for ds in DATASET_ORDER:
    row = {'BD': ds.capitalize()}
    for strat in STRATEGIES:
        key = (ds, strat)
        if key in all_results:
            res = all_results[key]
            avg = res['avg_client_metrics']
            row[f'ΔAcc_{strat}'] = round(avg['accuracy'] - NAYMA_RESULTS[ds]['accuracy'], 6)
            row[f'ΔF1_{strat}'] = round(avg['f1'] - NAYMA_RESULTS[ds]['f1'], 6)
        else:
            row[f'ΔAcc_{strat}'] = None
            row[f'ΔF1_{strat}'] = None
    diff_rows.append(row)

diff_cols = ['BD']
for strat in STRATEGIES:
    diff_cols += [f'ΔAcc_{strat}', f'ΔF1_{strat}']

df_diff = pd.DataFrame(diff_rows)[diff_cols]

print("\n" + "="*80)
print("📈 DIFFERENCES TABLE (Strategy - Nayma's PF)")
print("="*80)
print(df_diff.to_string(index=False))

df_diff.to_csv(out_dir / 'comparison_diff_vs_PF.csv', index=False)
print(f"\n✅ Differences saved to: {out_dir / 'comparison_diff_vs_PF.csv'}")